# Task manager with ipywidgets

This lab builds a task manager in the notebook: an entry and a button to add a task, the list of tasks, buttons to mark them done or remove them, and a filter (all, to do, done). It follows the course's Model-View-Controller organization: the model in a file, `model.py`, the view and the controller in the notebook's cells, with ipywidgets. The widgets show up under the cells, in Colab as in Jupyter or VS Code (the Codespace).

It is the Tkinter lab's twin, with the same model and the same controller: if you already wrote it, reuse your `model.py` and your `Controller` class. Only the view changes.

This is a linear lab: each part builds on the previous one. If you get stuck, run the solution and move on.

## 1. The model

Write `model.py`, plain Python with no interface: it holds the application's rules.

- `FILTERS = ("all", "to do", "done")`, the filters the interface will offer.
- `Task`, a dataclass: a `title` (`str`) and `done` (`bool`, `False` by default). The title loses its leading and trailing spaces, and an empty title raises a `ValueError`.
- `Checklist`, a dataclass whose only attribute, `tasks`, is a list of `Task`, empty at first, with four methods:
  - `add(title)` creates the task, appends it to the list and returns it; a title already there raises a `ValueError`;
  - `remove(task)` takes it out of the list;
  - `toggle(task)` marks it done if it was to do, and the other way round;
  - `filtered(which)` returns the list of the tasks that match one of the `FILTERS`, in the order they were added; any other filter raises a `ValueError`.

The `ValueError` messages are meant for the user: the interface will show them as they are.

The cell starts with `%%writefile model.py`: Jupyter writes the rest of the cell to that file instead of running it. The check cell, after the solution, tests your model.

**Useful documentation**

- [%%writefile](https://ipython.readthedocs.io/en/stable/interactive/magics.html#cellmagic-writefile)

In [ ]:
# Your code here

### Additional hints

The course on data classes shows the two tools `Task` and `Checklist` need: `__post_init__()`, which the dataclass calls after its generated `__init__`, validates the title (`self.title = self.title.strip()`, then `if not self.title: ...`); `field(default_factory=list)` gives each `Checklist` its own list. `any(...)` over a generator tells whether a task already has that title; compare with the cleaned-up title, `"  Bread "` is a duplicate of `"Bread"`. `toggle` is one line: `task.done = not task.done`.

```python
%%writefile model.py
from dataclasses import dataclass, field

FILTERS = ("all", "to do", "done")


@dataclass
class Task:
  title: str
  done: bool = False

  def __post_init__(self) -> None:
    self.title = self.title.strip()
    if not self.title:
      msg = "a task needs a title"
      raise ValueError(msg)


@dataclass
class Checklist:
  tasks: list[Task] = field(default_factory=list)

  def add(self, title: str) -> Task:
    task = Task(title)
    if any(...):
      msg = f"“{task.title}” is already in the list"
      raise ValueError(msg)
    ...

  def remove(self, task: Task) -> None: ...

  def toggle(self, task: Task) -> None: ...

  def filtered(self, which: str) -> list[Task]:
    if which == "all":
      return list(self.tasks)
    ...
```

### Solution

In [ ]:
%%writefile model.py
from dataclasses import dataclass, field

FILTERS = ("all", "to do", "done")


@dataclass
class Task:
  title: str
  done: bool = False

  def __post_init__(self) -> None:
    self.title = self.title.strip()
    if not self.title:
      msg = "a task needs a title"
      raise ValueError(msg)


@dataclass
class Checklist:
  tasks: list[Task] = field(default_factory=list)

  def add(self, title: str) -> Task:
    task = Task(title)
    if any(other.title == task.title for other in self.tasks):
      msg = f"“{task.title}” is already in the list"
      raise ValueError(msg)
    self.tasks.append(task)
    return task

  def remove(self, task: Task) -> None:
    self.tasks.remove(task)

  def toggle(self, task: Task) -> None:
    task.done = not task.done

  def filtered(self, which: str) -> list[Task]:
    if which == "all":
      return list(self.tasks)
    if which == "to do":
      return [task for task in self.tasks if not task.done]
    if which == "done":
      return [task for task in self.tasks if task.done]
    msg = f"unknown filter: {which!r}"
    raise ValueError(msg)

### Check

The next cell reloads `model.py` (otherwise the notebook keeps the version it imported first), then plays a short scenario. Expected results: `['Buy bread']` to do, `['Review the course']` done, then three `ValueError`s (empty title, duplicate, unknown filter), and finally `['Buy bread']` after the removal.

In [ ]:
import importlib

import model

importlib.reload(model)

checklist = model.Checklist()
checklist.add("Buy bread")
review = checklist.add("  Review the course ")
checklist.toggle(review)
print("To do:", [task.title for task in checklist.filtered("to do")])
print("Done:", [task.title for task in checklist.filtered("done")])

for attempt in (
  lambda: checklist.add("   "),
  lambda: checklist.add("Buy bread"),
  lambda: checklist.filtered("urgent"),
):
  try:
    attempt()
  except ValueError as error:
    print("ValueError:", error)
  else:
    print("No ValueError: to fix")

checklist.remove(review)
print("All:", [task.title for task in checklist.filtered("all")])

## 2. The view

Write a `TaskView` class, like the course's ipywidgets view. It knows nothing of the model: all it imports from `model.py` is `FILTERS`. It creates these widgets, in its attributes:

- `entry`, a `Text`, and the `add` button ("Add"), on one row;
- `tasks`, a 10-row `SelectMultiple`, empty at first;
- on one row, the `toggle` ("Done / to do") and `remove` ("Remove") buttons, and `which`, a `Dropdown` of the `FILTERS`;
- `status`, a `Label`, and `message`, an `HTML` for the errors.

It puts them in `root`, a `VBox`, which the notebook will display. The check at the end uses these attribute names: keep them. The controller won't touch them: it will only use the view's methods, the same as the Tkinter lab's view.

The first methods read or change the display:

- `show(lines, status)` replaces the list's content with `lines`, a list of strings, `status`'s text with `status`, and clears the message;
- `selection()` returns the list of the selected lines' indices;
- `typed_title()` returns the entry's text, `clear_entry()` clears it, and `chosen_filter()` returns the chosen filter;
- `report(message)` shows the message in bold red in `message`.

The others connect the widgets to an action, a function without arguments that the controller will give them, like `on_plus(action)` in the course's ipywidgets view:

- `on_add(action)`: the "Add" button;
- `on_toggle(action)`: the "Done / to do" button;
- `on_remove(action)`: the "Remove" button;
- `on_filter(action)`: choosing a filter.

The cell after the solution displays a preview of the view, with two sample lines: its buttons do nothing yet.

**Useful documentation**

- [SelectMultiple](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html#selectmultiple)
- [HTML](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html#html)

In [ ]:
# Your code here

### Additional hints

- `widgets.SelectMultiple(options=[], rows=10)`: `rows` sets its height, in lines. Ctrl+click and Shift+click select several lines.
- `widgets.Dropdown(options=FILTERS)` selects the first option at first.
- `root` is a `VBox` of `HBox`es and widgets: `widgets.VBox([widgets.HBox([self.entry, self.add]), self.tasks, ...])`.
- A list is replaced as a whole: `self.tasks.options = lines`. Its `index` is the tuple of the selected indices.
- An `HTML` widget shows HTML: `f"<b style='color: crimson'>{message}</b>"`.
- `on_click` passes the clicked button to the callback, which the action doesn't need: `self.add.on_click(lambda button: action())`. `observe` passes it a dict describing the change: `self.which.observe(lambda change: action(), names="value")`.

```python
from collections.abc import Callable

import ipywidgets as widgets
from IPython.display import display

from model import FILTERS


class TaskView:
  def __init__(self) -> None:
    self.entry = widgets.Text(placeholder="New task")
    self.add = widgets.Button(description="Add")
    self.tasks = ...
    self.toggle = widgets.Button(description="Done / to do")
    ...
    self.status = widgets.Label()
    self.message = widgets.HTML()
    self.root = widgets.VBox([widgets.HBox([self.entry, self.add]), ...])

  def show(self, lines: list[str], status: str) -> None:
    self.tasks.options = lines
    ...

  def selection(self) -> list[int]:
    return ...

  def typed_title(self) -> str:
    return self.entry.value

  def clear_entry(self) -> None: ...

  def chosen_filter(self) -> str: ...

  def report(self, message: str) -> None: ...

  def on_add(self, action: Callable[[], None]) -> None:
    self.add.on_click(lambda button: action())

  def on_toggle(self, action: Callable[[], None]) -> None: ...

  def on_remove(self, action: Callable[[], None]) -> None: ...

  def on_filter(self, action: Callable[[], None]) -> None: ...
```

### Solution

In [ ]:
from collections.abc import Callable

import ipywidgets as widgets
from IPython.display import display

from model import FILTERS


class TaskView:
  def __init__(self) -> None:
    self.entry = widgets.Text(placeholder="New task")
    self.add = widgets.Button(description="Add")
    self.tasks = widgets.SelectMultiple(options=[], rows=10)
    self.toggle = widgets.Button(description="Done / to do")
    self.remove = widgets.Button(description="Remove")
    self.which = widgets.Dropdown(options=FILTERS)
    self.status = widgets.Label()
    self.message = widgets.HTML()
    self.root = widgets.VBox(
      [
        widgets.HBox([self.entry, self.add]),
        self.tasks,
        widgets.HBox([self.toggle, self.remove, self.which]),
        self.status,
        self.message,
      ]
    )

  def show(self, lines: list[str], status: str) -> None:
    self.tasks.options = lines
    self.status.value = status
    self.message.value = ""

  def selection(self) -> list[int]:
    return list(self.tasks.index)

  def typed_title(self) -> str:
    return self.entry.value

  def clear_entry(self) -> None:
    self.entry.value = ""

  def chosen_filter(self) -> str:
    return self.which.value

  def report(self, message: str) -> None:
    self.message.value = f"<b style='color: crimson'>{message}</b>"

  def on_add(self, action: Callable[[], None]) -> None:
    self.add.on_click(lambda button: action())

  def on_toggle(self, action: Callable[[], None]) -> None:
    self.toggle.on_click(lambda button: action())

  def on_remove(self, action: Callable[[], None]) -> None:
    self.remove.on_click(lambda button: action())

  def on_filter(self, action: Callable[[], None]) -> None:
    self.which.observe(lambda change: action(), names="value")

### Preview

In [ ]:
preview = TaskView()
preview.show(["[ ] Buy bread", "[x] Review the course"], "1 task(s) to do")
preview.report("Sample error")
display(preview.root)

## 3. The controller

Write a `Controller` class that receives the model and the view, gives its methods to the view through the `on_…` methods and keeps the view up to date. The controller touches no widget: it only goes through the view's methods. It is the Tkinter lab's `Controller` class: if you wrote it, reuse it as is.

- `refresh()` asks the model for the chosen filter's tasks, keeps them in `self.visible`, and shows them, one line per task, as `[ ] title` (to do) or `[x] title` (done), with the number of tasks to do in `status`. Line `i` of the list is task `self.visible[i]`.
- `add()` adds the task typed in. If the model raises a `ValueError`, the view reports its message; otherwise, the entry is cleared and the list refreshed. The view calls it on the "Add" button.
- `toggle()` and `remove()` act on the selected tasks, then refresh the list.
- Choosing a filter refreshes the list.

The cell after the solution creates the application and displays it.

In [ ]:
# Your code here

### Additional hints

- The controller gives its methods to the view without calling them: `view.on_add(self.add)`.
- It reads the entry and the filter through the view, `self.view.typed_title()` and `self.view.chosen_filter()`, never through its widgets.
- `try` / `except ValueError as error` / `else`, as in the course's controller: the `else` only runs if adding worked.
- The list only shows the current filter's tasks: its line `i` isn't `model.tasks[i]`, hence `self.visible`. A `selected()` method returning `[self.visible[i] for i in self.view.selection()]` serves both `toggle` and `remove`.
- It is the Tkinter lab's controller, without a single change: `on_click` and `observe` are in the ipywidgets view, as `command=` and `bind` are in the Tkinter view.

```python
from model import Checklist, Task


class Controller:
  def __init__(self, model: Checklist, view: TaskView) -> None:
    self.model = model
    self.view = view
    self.visible: list[Task] = []
    view.on_add(self.add)
    ...
    self.refresh()

  def refresh(self) -> None:
    self.visible = self.model.filtered(self.view.chosen_filter())
    lines = []
    for task in self.visible:
      ...
    left = ...
    self.view.show(lines, f"{left} task(s) to do")

  def add(self) -> None:
    try:
      ...
    except ValueError as error:
      ...
    else:
      ...

  def selected(self) -> list[Task]:
    return [self.visible[index] for index in self.view.selection()]

  def toggle(self) -> None: ...

  def remove(self) -> None: ...
```

### Solution

In [ ]:
from model import Checklist, Task


class Controller:
  def __init__(self, model: Checklist, view: TaskView) -> None:
    self.model = model
    self.view = view
    self.visible: list[Task] = []
    view.on_add(self.add)
    view.on_toggle(self.toggle)
    view.on_remove(self.remove)
    view.on_filter(self.refresh)
    self.refresh()

  def refresh(self) -> None:
    self.visible = self.model.filtered(self.view.chosen_filter())
    lines = []
    for task in self.visible:
      box = "[x]" if task.done else "[ ]"
      lines.append(f"{box} {task.title}")
    left = len(self.model.filtered("to do"))
    self.view.show(lines, f"{left} task(s) to do")

  def add(self) -> None:
    try:
      self.model.add(self.view.typed_title())
    except ValueError as error:
      self.view.report(str(error))
    else:
      self.view.clear_entry()
      self.refresh()

  def selected(self) -> list[Task]:
    return [self.visible[index] for index in self.view.selection()]

  def toggle(self) -> None:
    for task in self.selected():
      self.model.toggle(task)
    self.refresh()

  def remove(self) -> None:
    for task in self.selected():
      self.model.remove(task)
    self.refresh()

### The application

Try it: add tasks, then an empty title and a duplicate; toggle several tasks at once (Ctrl+click); change the filter; remove some.

In [ ]:
view = TaskView()
controller = Controller(Checklist(), view)
display(view.root)

### Check without a mouse

The next cell clicks for you, with the buttons' `click()` method, on the application displayed above. Expected results: two tasks to do and the duplicate's message, then `[x] Review the course` done, alone in the `done` filter, and finally `[ ] Buy bread` alone after its removal.

In [ ]:
view.entry.value = "Buy bread"
view.add.click()
view.entry.value = "Review the course"
view.add.click()
view.entry.value = "Buy bread"
view.add.click()
print(view.tasks.options, view.status.value, view.message.value)

view.tasks.value = ("[ ] Review the course",)
view.toggle.click()
print(view.tasks.options, view.status.value)

view.which.value = "done"
print(view.tasks.options)
view.tasks.value = view.tasks.options
view.remove.click()
view.which.value = "all"
print(view.tasks.options, view.status.value)